In [0]:
#Bibliotecas 
from pyspark.sql import functions as F

In [0]:
#Tratamento das tabelas
 
PARQUET_PATH = "/Volumes/workspace/olist_bronze/parquet"

tables = [
    "olist_customers_dataset",
    "olist_orders_dataset",
    "olist_order_items_dataset",
    "olist_order_payments_dataset",
    "olist_order_reviews_dataset",
    "olist_products_dataset",
    "olist_sellers_dataset",
    "olist_geolocation_dataset",
    "product_category_name_translation"
]

dataframes = {}

for table in tables:
    df = spark.read.parquet(f"{PARQUET_PATH}/{table}")
    dataframes[table] = df

# Diagnóstico de valores nulos
null_results = []

for table, df in dataframes.items():

    total_rows = df.count()

    for column in df.columns:

        null_count = df.filter(
            F.col(column).isNull()
        ).count()

        null_results.append((
            table,
            column,
            total_rows,
            null_count,
            round(null_count / total_rows * 100, 2)
            if total_rows > 0 else 0
        ))

null_report = spark.createDataFrame(
    null_results,
    ["table_name", "column_name", "total_rows",
     "null_count", "null_percentage"]
)

display(
    null_report
    .filter(F.col("null_count") > 0)
    .orderBy(F.desc("null_percentage"))
)


In [0]:
""" 
Nulos esperados na base de dados:
Tabela/coluna	                        Nulos	Interpretação	
reviews.review_comment_title	        88,47%	Cliente não informou título	
reviews.review_comment_message	        60,56%	Cliente não escreveu comentário	
orders.order_delivered_customer_date	2,98%	Pedido pode não ter sido entregue	
orders.order_delivered_carrier_date	    1,79%	Pedido pode não ter sido enviado	
orders.order_approved_at	            0,16%	Aprovação ausente	
products.product_category_name	        1,85%	Produto sem categoria informada
"""

In [0]:
# Registra as tabelas como views temporárias
for table, df in dataframes.items():
    df.createOrReplaceTempView(table)

# # Excluir os registros que tiver o review_creation_date NULL (8.764 registros)
# display(
#     spark.sql("""
#         SELECT COUNT(*)
#         FROM olist_order_reviews_dataset
#         WHERE review_creation_date IS NULL
#     """)
# )

# Excluir os registros quando order_status = 'delivered' AND order_delivered_customer_date IS NULL (8 registros)
# display(
#     spark.sql("""
#         SELECT *
#         FROM olist_orders_dataset
#         WHERE order_status = 'delivered' AND order_delivered_customer_date IS NULL
#     """)
# )

# Excluir os registros que tiver o product_category_name NULL (610 registros)
# display(
#     spark.sql("""
#         SELECT *
#         FROM olist_products_dataset
#         WHERE product_category_name IS NULL
#     """)
# )


In [0]:
SILVER_PATH = "/Volumes/workspace/olist_silver/parquet"

dataframes_silver = {}

# Copia os dados de todas as tabelas usando Spark SQL
for table in dataframes.keys():
    dataframes_silver[table] = spark.sql(f"""
        SELECT *
        FROM {table}
    """)

# Regra 0: remover reviews sem data de criação
dataframes_silver["olist_order_reviews_dataset"] = spark.sql("""
    SELECT *
    FROM olist_order_reviews_dataset
    WHERE NOT (
        review_creation_date IS NULL
    )
""")

# Regra 1: remover pedidos entregues sem data de entrega
dataframes_silver["olist_orders_dataset"] = spark.sql("""
    SELECT *
    FROM olist_orders_dataset
    WHERE NOT (
        order_status = 'delivered'
        AND order_delivered_customer_date IS NULL
    )
""")

# Regra 2: remover produtos com a categoria vazia
dataframes_silver["olist_products_dataset"] = spark.sql("""
    SELECT *
    FROM olist_products_dataset
    WHERE NOT (
        product_category_name IS NULL
    )
""")

In [0]:
for table, df in dataframes_silver.items():

    output_path = f"{SILVER_PATH}/{table}"

    df.write \
        .format("parquet") \
        .mode("overwrite") \
        .save(output_path)

    print(f"Tabela {table} gravada com sucesso!")

In [0]:
for table in dataframes_silver.keys():

    df_check = spark.read.parquet(
        f"{SILVER_PATH}/{table}"
    )

    print(
        f"{table}: {df_check.count()} registros"
    )

In [0]:
display(
    dataframes_silver["olist_orders_dataset"]
    .groupBy("order_status")
    .count()
    .orderBy("order_status")
)